Import

In [3]:
import cv2
import numpy as np
from PIL import Image
import tensorflow as tf
from tflite_support import metadata
from tflite_model_maker import object_detector
import os

c:\Users\march\miniconda3\envs\tensorflow\lib\site-packages\tensorflow_addons\utils\tfa_eol_msg.py:23: UserWarning: 

TensorFlow Addons (TFA) has ended development and introduction of new features.
TFA has entered a minimal maintenance and release mode until a planned end of life in May 2024.
Please modify downstream libraries to take dependencies from other repositories in our TensorFlow community (e.g. Keras, Keras-CV, and Keras-NLP). 

For more information see: https://github.com/tensorflow/addons/issues/2807 

  warnings.warn(
c:\Users\march\miniconda3\envs\tensorflow\lib\site-packages\tensorflow_addons\utils\ensure_tf_install.py:53: UserWarning: Tensorflow Addons supports using Python ops for all Tensorflow versions above or equal to 2.12.0 and strictly below 2.15.0 (nightly versions are not supported). 
 The versions of TensorFlow you are currently using is 2.8.0 and is not supported. 
Some things might work, some things might not.
If you were to encounter a bug, do not file an i

Import Data Set

In [4]:
split_dirs = {
    'train': {'images': 'split/50m/train/images', 'annotations': 'split/50m/train/annotations'},
    'val': {'images': 'split/50m/val/images', 'annotations': 'split/50m/val/annotations'},
}

# Load the data for each split using Pascal VOC format
train_data = object_detector.DataLoader.from_pascal_voc(
    split_dirs['train']['images'], 
    split_dirs['train']['annotations'], 
    label_map={1: "mobil"}
)
val_data = object_detector.DataLoader.from_pascal_voc(
    split_dirs['val']['images'], 
    split_dirs['val']['annotations'], 
    label_map={1: "mobil"}
)

INFO:tensorflow:Cache will be stored in C:\Users\march\AppData\Local\Temp\tmplu9_zc2n with prefix filename fd2c112752054ea1fcebcf49d7779205. Cache_prefix is C:\Users\march\AppData\Local\Temp\tmplu9_zc2n\fd2c112752054ea1fcebcf49d7779205
INFO:tensorflow:On image 0
INFO:tensorflow:On image 100
INFO:tensorflow:On image 200
INFO:tensorflow:Cache will be stored in C:\Users\march\AppData\Local\Temp\tmpxdwtxwkg with prefix filename fd2c112752054ea1fcebcf49d7779205. Cache_prefix is C:\Users\march\AppData\Local\Temp\tmpxdwtxwkg\fd2c112752054ea1fcebcf49d7779205
INFO:tensorflow:On image 0


Test Dataset

In [5]:
test_dirs = {
    'test': {'images': 'split/50m/test/images', 'annotations': 'split/50m/test/annotations'}
}

test_data = object_detector.DataLoader.from_pascal_voc(
    test_dirs['test']['images'], 
    test_dirs['test']['annotations'], 
    label_map={1: "mobil"}
)

INFO:tensorflow:Cache will be stored in C:\Users\march\AppData\Local\Temp\tmpm057_v4v with prefix filename fd2c112752054ea1fcebcf49d7779205. Cache_prefix is C:\Users\march\AppData\Local\Temp\tmpm057_v4v\fd2c112752054ea1fcebcf49d7779205
INFO:tensorflow:On image 0


Reload TFLite

In [6]:
model_path = os.path.join("MyModel","50m","model.tflite")
# Load the TFLite model
interpreter = tf.lite.Interpreter(model_path=model_path)
interpreter.allocate_tensors()

Buat Print Foto

In [1]:
def preprocess_image(image_path, input_size):
# """Preprocess the input image to feed to the TFLite model"""
    img = tf.io.read_file(image_path)
    img = tf.io.decode_image(img, channels=3)
    img = tf.image.convert_image_dtype(img, tf.uint8)
    original_image = img
    resized_img = tf.image.resize(img, input_size)
    resized_img = resized_img[tf.newaxis, :]
    resized_img = tf.cast(resized_img, dtype=tf.uint8)
    return resized_img, original_image

In [2]:
def detect_objects(interpreter, image, threshold):
# """Returns a list of detection results, each a dictionary of object info."""

    signature_fn = interpreter.get_signature_runner()

    # Feed the input image to the model
    output = signature_fn(images=image)

    # Get all outputs from the model
    count = int(np.squeeze(output['output_0']))
    scores = np.squeeze(output['output_1'])
    classes = np.squeeze(output['output_2'])
    boxes = np.squeeze(output['output_3'])

    results = []
    for i in range(count):
      if scores[i] >= threshold:
        result = {
          'bounding_box': boxes[i],
          'class_id': classes[i],
          'score': scores[i]
        }
        results.append(result)
    return results

In [4]:
def run_odt_and_draw_results(image_path, interpreter, threshold=0.5):
# """Run object detection on the input image and draw the detection results"""
    # Load the input shape required by the model
    _, input_height, input_width, _ = interpreter.get_input_details()[0]['shape']

    # Load the input image and preprocess it
    preprocessed_image, original_image = preprocess_image(
        image_path,
        (input_height, input_width)
      )

    # Run object detection on the input image
    results = detect_objects(interpreter, preprocessed_image, threshold=threshold)

    # Plot the detection results on the input image
    original_image_np = original_image.numpy().astype(np.uint8)
    for obj in results:
      # Convert the object bounding box from relative coordinates to absolute
      ymin, xmin, ymax, xmax = obj['bounding_box']
      xmin = int(xmin * original_image_np.shape[1])
      xmax = int(xmax * original_image_np.shape[1])
      ymin = int(ymin * original_image_np.shape[0])
      ymax = int(ymax * original_image_np.shape[0])

      # Find the class index of the current object
      class_id = int(obj['class_id'])
      # Draw the bounding box and label on the image
      color = [int(c) for c in COLORS[class_id]]
      cv2.rectangle(original_image_np, (xmin, ymin), (xmax, ymax), color, 2)
      # Make adjustments to make the label visible for all objects
      y = ymin - 15 if ymin - 15 > 15 else ymin + 15
      label = "{}: {:.0f}%".format(classes[class_id], obj['score'] * 100)
      cv2.putText(original_image_np, label, (xmin, y), cv2.FONT_HERSHEY_SIMPLEX, 0.5, color, 2)

    # Return the final image
    original_uint8 = original_image_np.astype(np.uint8)
    return original_uint8

In [13]:
displayer = metadata.MetadataDisplayer.with_model_file(model_path)

# Load label list from metadata.
file_name = displayer.get_packed_associated_file_list()[0]
label_map_file = displayer.get_associated_file_buffer(file_name).decode()
label_list = list(filter(lambda x: len(x) > 0, label_map_file.splitlines()))

# Load labels (if available)
num_classes = len(label_list)
classes = ['???'] * num_classes
for label_id, label_name in enumerate(label_list):
    classes[label_id] = label_name

# Define a list of colors for visualization
COLORS = np.random.randint(0, 255, size=(len(classes), 3), dtype=np.uint8)


Print Test Data (Image)

In [5]:
model_path = os.path.join("MyModel","20m","model.tflite")
# Load the TFLite model
interpreter = tf.lite.Interpreter(model_path=model_path)
interpreter.allocate_tensors()

In [ ]:
# INPUT_IMAGE_URL = "split/50m/test/images/1-33_png.rf.d65b76b13b5f052136cf136f170f6ccc.jpg"
INPUT_IMAGE_URL = "split/10-25m/test/images/15M-9-_jpg.rf.c8ac478142185d16de4b6741768d768f.jpg"
# INPUT_IMAGE_URL = "split/300m/test/images/0-cka-147_jpg.rf.d1136a5a3a4b60c7436bc4d3bffe484e.jpg"
DETECTION_THRESHOLD = 0.1

# Run inference and draw detection result on the local copy of the original file
detection_result_image = run_odt_and_draw_results(
    INPUT_IMAGE_URL,
    interpreter,
    threshold=DETECTION_THRESHOLD
)
detection_result_image_rgb = cv2.cvtColor(detection_result_image, cv2.COLOR_BGR2RGB)
cv2.imwrite("10-25mCek.png", detection_result_image_rgb, [cv2.IMWRITE_PNG_COMPRESSION, 0])

True

In [83]:
import os
import cv2
import numpy as np

# Define the input dataset folder and threshold
DATASET_FOLDER = "split/50m/test/images"
DETECTION_THRESHOLD = 0.5
OUTPUT_FOLDER = "Eval_Result/50m/50m"  # Folder to save the output images

# Create the output folder if it doesn't exist
if not os.path.exists(OUTPUT_FOLDER):
    os.makedirs(OUTPUT_FOLDER)

# Function to run object detection on multiple images
def process_dataset(dataset_folder, interpreter, detection_threshold):
    # Get all image files in the dataset folder
    image_files = [f for f in os.listdir(dataset_folder) if f.endswith(('.jpg', '.png', '.jpeg'))]
    
    # Iterate through each image in the dataset
    for image_file in image_files:
        image_path = os.path.join(dataset_folder, image_file)
        
        # Run inference and draw the detection result
        detection_result_image = run_odt_and_draw_results(image_path, interpreter, threshold=detection_threshold)
        
        # Convert the result to RGB (for correct color display)
        detection_result_image_rgb = cv2.cvtColor(detection_result_image, cv2.COLOR_BGR2RGB)
        
        # Save the output image in the designated folder
        output_image_path = os.path.join(OUTPUT_FOLDER, f"result_{image_file}")
        cv2.imwrite(output_image_path, detection_result_image_rgb, [cv2.IMWRITE_PNG_COMPRESSION, 0])
        print(f"Saved detection result for {image_file} to {output_image_path}")

# Example usage
process_dataset(DATASET_FOLDER, interpreter, DETECTION_THRESHOLD)


Saved detection result for 1-33_png.rf.d65b76b13b5f052136cf136f170f6ccc.jpg to Eval_Result/50m/50m\result_1-33_png.rf.d65b76b13b5f052136cf136f170f6ccc.jpg
Saved detection result for 10-33_png.rf.ec6e20a50c31c14c6f875d5bcd0fb87d.jpg to Eval_Result/50m/50m\result_10-33_png.rf.ec6e20a50c31c14c6f875d5bcd0fb87d.jpg
Saved detection result for 14-33_png.rf.a70183acd80e2631c5d6ead5233b77fe.jpg to Eval_Result/50m/50m\result_14-33_png.rf.a70183acd80e2631c5d6ead5233b77fe.jpg
Saved detection result for 15-67_png.rf.ab0a6f1378d4f280e16ac6d1411bc594.jpg to Eval_Result/50m/50m\result_15-67_png.rf.ab0a6f1378d4f280e16ac6d1411bc594.jpg
Saved detection result for 16-00_png.rf.0187415fc709be1138b76979c38d7ef5.jpg to Eval_Result/50m/50m\result_16-00_png.rf.0187415fc709be1138b76979c38d7ef5.jpg
Saved detection result for 16-33_png.rf.6550b03c7326e162cf98a75ada361f32.jpg to Eval_Result/50m/50m\result_16-33_png.rf.6550b03c7326e162cf98a75ada361f32.jpg


KeyboardInterrupt: 